# CloudDenseNet — DenseNet121 Transfer Learning on Filtered NASA GLOBE (Colab)

Colab adaptation of `14_transfer_learning_clouddensenet.ipynb`.

Replication of *CloudDenseNet: Lightweight Ground-Based Cloud Classification*
(Li et al., Sensors 2023).

**What this notebook does:**
- Loads pre-filtered NASA GLOBE images from Google Drive
- Fine-tunes DenseNet121 (ImageNet weights) using the paper's exact training schedule:
  a custom **TopBlock** head, **Focal Loss**, and **5-phase gradual unfreezing**

**Training schedule** — paper §4:

| Phase | Layers unfrozen | Head LR | Backbone LR | Epochs |
|-------|----------------|---------|-------------|--------|
| 1 | TopBlock only | 1e-4 | — | 10 |
| 2 | TopBlock only | 1e-5 | — | 10 |
| 3 | TopBlock + DenseBlock3 onwards | 1e-5 | 5e-5 | 20 |
| 4 | TopBlock + DenseBlock3 onwards | 1e-5 | 1e-5 | 20 |
| 5 | All layers | 1e-6 | 1e-6 | 20 |

In [ ]:
import os
from google.colab import drive
from tqdm.auto import tqdm

unzipped_images_dir = '/content/my_images/extract_filtered'
zip_path = '/content/drive/MyDrive/Colab Notebooks/extract_filtered.zip'
expected_image_count = 30000

def count_files_with_progress(directory, description):
    file_count = 0
    for root, dirs, files in tqdm(os.walk(directory), desc=description, unit="dir"):
        file_count += len(files)
    return file_count

if os.path.exists(unzipped_images_dir) and os.listdir(unzipped_images_dir):
    current_image_count = count_files_with_progress(unzipped_images_dir, "Counting existing images")
    if current_image_count == expected_image_count:
        print(f"✅ Images already extracted to {unzipped_images_dir}. Skipping extraction. ({current_image_count} files found)")
    else:
        print(f"⚠️ Images directory {unzipped_images_dir} exists but contains {current_image_count} files instead of the expected {expected_image_count}.")
        user_response = input("Do you want to re-extract the images? (yes/no): ").strip().lower()
        if user_response == 'yes':
            print("Proceeding with re-extraction...")
            drive.mount('/content/drive')
            if os.path.exists(zip_path):
                print("✅ Zip file found! Starting re-extraction...")
                os.makedirs('/content/my_images', exist_ok=True)
                !unzip -o -q "{zip_path}" -d /content/my_images
                final_image_count = count_files_with_progress(unzipped_images_dir, "Counting re-extracted images")
                if final_image_count == expected_image_count:
                    print(f"🎉 Re-extraction complete and verified! ({final_image_count} files found)")
                else:
                    print(f"❌ Re-extraction complete, but found {final_image_count} files instead of {expected_image_count}.")
            else:
                print(f"❌ Error: Zip file not found at {zip_path}")
        else:
            print(f"Skipping re-extraction. Using existing images ({current_image_count} files).")
else:
    print(f"Images not found in {unzipped_images_dir}. Proceeding with extraction...")
    drive.mount('/content/drive')
    if os.path.exists(zip_path):
        print("✅ Zip file found! Starting extraction...")
        os.makedirs('/content/my_images', exist_ok=True)
        !unzip -q "{zip_path}" -d /content/my_images
        final_image_count = count_files_with_progress(unzipped_images_dir, "Counting extracted images")
        if final_image_count == expected_image_count:
            print(f"🎉 Extraction complete and verified! ({final_image_count} files found)")
        else:
            print(f"❌ Extraction complete, but found {final_image_count} files instead of {expected_image_count}.")
    else:
        print(f"❌ Error: Zip file not found at {zip_path}")
        print("Double check if there is a typo in 'Colab Notebooks' or the filename.")

In [ ]:
from pathlib import Path
from collections import Counter

IMAGES_PATH = Path("/content/my_images/extract_filtered")

# The 10 cloud types replicated from the paper (excludes any extra classes in the zip)
CLOUD_TYPES = ["Ac", "As", "Cb", "Cc", "Ci", "Cs", "Cu", "Ns", "Sc", "St"]

def index_labeled_images(images_path=IMAGES_PATH, cloud_types=None, max_per_class=3000):
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images

    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        if cloud_types is not None and cloud_dir.name not in cloud_types:
            continue
        count = 0
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            if count >= max_per_class:
                break
            labeled_images[img_path.name] = {
                "label": cloud_dir.name,
                "path": str(img_path)
            }
            count += 1

    return labeled_images

labeled_images = index_labeled_images(cloud_types=CLOUD_TYPES)

counts = Counter(v["label"] for v in labeled_images.values())
for folder in sorted(counts):
    print(f"  {folder:6s}  {counts[folder]:,}")
print(f"  {'TOTAL':6s}  {sum(counts.values()):,}")

In [ ]:
import numpy as np

def extract_labels(labeled_images):
    paths, labels = [], []
    for image_name in labeled_images:
        paths.append(labeled_images[image_name]["path"])
        labels.append(labeled_images[image_name]["label"])
    return paths, np.array(labels)

paths, labels = extract_labels(labeled_images)

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"device: {device}")

In [ ]:
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (224, 224)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0,
    translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR,
    fill=0
)

class WrapTranslation:
    """Circular shift by a random fraction of the image size — picklable for num_workers > 0."""
    def __init__(self, max_frac):
        self.max_frac = max_frac

    def __call__(self, x):
        h, w = x.shape[-2], x.shape[-1]
        shift_h = int(torch.randint(-int(self.max_frac * h), int(self.max_frac * h) + 1, (1,)).item())
        shift_w = int(torch.randint(-int(self.max_frac * w), int(self.max_frac * w) + 1, (1,)).item())
        return torch.roll(x, shifts=(shift_h, shift_w), dims=(-2, -1))

wrap_translation = WrapTranslation(MAX_FRAC)

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(
        degrees=20, scale=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR, fill=0
    ),
    T.RandomResizedCrop(
        size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10),
        interpolation=InterpolationMode.BILINEAR
    ),
])

one_of = T.RandomChoice([
    wrap_translation,
    border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

train_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
    normalize,
])

eval_transforms = T.Compose([
    T.Resize(IMG_SIZE, interpolation=InterpolationMode.BILINEAR),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    normalize,
])

In [ ]:
import torchvision
import torch.nn as nn

weights = torchvision.models.DenseNet121_Weights.IMAGENET1K_V1
model = torchvision.models.densenet121(weights=weights).to(device)
print("DenseNet121 loaded")

In [ ]:
from sklearn.preprocessing import LabelEncoder

def encode_labels(labels):
    ordinal_encoder = LabelEncoder()
    encoded_labels = ordinal_encoder.fit_transform(labels)
    return encoded_labels, ordinal_encoder.classes_

encoded_labels, class_names = encode_labels(labels)
print(f"Classes ({len(class_names)}): {class_names}")

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedShuffleSplit
from PIL import Image

class MyImages(Dataset):
    def __init__(self, paths, encoded_labels, split="train", test_size=0.2, val_size=0.1,
                 random_state=42, transform=None, split_indices=None):
        if split not in {None, "train", "val", "test"}:
            raise ValueError(f"split must be one of {{'None','train','val','test'}}, got {split!r}")

        self.transform = transform
        paths = np.array(list(paths))
        encoded_labels = np.array(list(encoded_labels))

        if split_indices is None:
            sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
            trainval_idx, test_idx = next(sss1.split(paths, encoded_labels))

            val_within_trainval = val_size / (1.0 - test_size)
            sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_within_trainval, random_state=random_state)
            train_rel_idx, val_rel_idx = next(sss2.split(paths[trainval_idx], encoded_labels[trainval_idx]))

            split_indices = {
                "train": trainval_idx[train_rel_idx],
                "val":   trainval_idx[val_rel_idx],
                "test":  test_idx,
            }

        idx = split_indices[split]
        self.paths = paths[idx].tolist()
        self.encoded_labels = encoded_labels[idx].tolist()

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.encoded_labels[idx]


paths_arr  = np.array(list(paths))
labels_arr = np.array(list(encoded_labels))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
trainval_idx, test_idx = next(sss1.split(paths_arr, labels_arr))

sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.1/0.8, random_state=42)
train_rel_idx, val_rel_idx = next(sss2.split(paths_arr[trainval_idx], labels_arr[trainval_idx]))

split_indices = {
    "train": trainval_idx[train_rel_idx],
    "val":   trainval_idx[val_rel_idx],
    "test":  test_idx,
}

train_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="train", transform=train_transforms,
                     split_indices=split_indices)
valid_set = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="val",   transform=eval_transforms,
                     split_indices=split_indices)
test_set  = MyImages(paths=paths, encoded_labels=encoded_labels,
                     split="test",  transform=eval_transforms,
                     split_indices=split_indices)

print(f"train: {len(train_set):,}  val: {len(valid_set):,}  test: {len(test_set):,}")

In [ ]:
from torch.utils.data import DataLoader, WeightedRandomSampler

train_labels_list = [train_set.encoded_labels[i] for i in range(len(train_set))]
class_counts  = np.bincount(train_labels_list)
class_weights = 1.0 / class_counts
sample_weights = torch.tensor([class_weights[l] for l in train_labels_list], dtype=torch.float)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_set, batch_size=64, sampler=sampler, num_workers=2)
valid_loader = DataLoader(valid_set, batch_size=64, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=64, num_workers=2)

In [ ]:
n_classes = len(class_names)


class TopBlock(nn.Module):
    """
    Custom classification head from CloudDenseNet (Li et al., Sensors 2023).

    Replaces DenseNet121's single linear classifier with a 2-layer MLP:
      BN -> Dropout -> Linear(1024, hidden) -> ReLU -> BN -> Dropout -> Linear(hidden, n_classes)

    Weights are initialised with LeCun uniform distribution, as specified in the paper.
    """
    def __init__(self, in_features: int, n_classes: int,
                 hidden_dim: int = 512, dropout: float = 0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Dropout(p=dropout),
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(p=dropout / 2),
            nn.Linear(hidden_dim, n_classes),
        )
        self._lecun_init()

    def _lecun_init(self):
        for m in self.block:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.block(x)


for param in model.parameters():
    param.requires_grad = False

model.classifier = TopBlock(
    in_features=1024,
    n_classes=n_classes,
).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Classes ({n_classes}): {class_names}")
print(f"Trainable params: {trainable:,} / {total:,}")

In [ ]:
!pip install torchmetrics

In [ ]:
import torchmetrics


def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            metric.update(model(X_batch), y_batch)
    return metric.compute()

accuracy = torchmetrics.Accuracy(task='multiclass', num_classes=n_classes).to(device)

In [ ]:
import torch.nn.functional as F


class FocalLoss(nn.Module):
    """
    Focal Loss (Lin et al., 2017).

    Down-weights easy (high-confidence) examples so the model focuses on
    hard, misclassified ones.  gamma=0 reduces to standard cross-entropy.
    Used by the paper in place of cross-entropy.
    """
    def __init__(self, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)
        loss = ((1.0 - p_t) ** self.gamma) * ce
        return loss.mean() if self.reduction == 'mean' else loss.sum()


focal_loss = FocalLoss(gamma=2.0).to(device)
print("FocalLoss ready (gamma=2.0)")

In [ ]:
def train_phase(model, optimizer, loss_fn, metric,
                train_loader, valid_loader,
                n_epochs, patience, checkpoint_path, phase_label=''):
    """
    Run one phase of the gradual-unfreeze schedule.

    Keeps frozen BatchNorm layers in eval mode so their running statistics
    are not corrupted; only BN layers with requires_grad=True are trained.
    Best weights are saved to checkpoint_path and restored before returning.
    """
    history    = {'train_losses': [], 'train_metrics': [], 'valid_metrics': []}
    best_val   = 0.0
    no_improve = 0

    for epoch in range(n_epochs):
        model.eval()
        model.classifier.train()
        for module in model.modules():
            if isinstance(module, (nn.BatchNorm2d, nn.BatchNorm1d)):
                if any(p.requires_grad for p in module.parameters()):
                    module.train()

        total_loss = 0.0
        metric.reset()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss   = loss_fn(y_pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            metric.update(y_pred, y_batch)

        train_loss = total_loss / len(train_loader)
        train_acc  = metric.compute().item()
        val_acc    = evaluate_tm(model, valid_loader, metric).item()

        history['train_losses'].append(train_loss)
        history['train_metrics'].append(train_acc)
        history['valid_metrics'].append(val_acc)

        star = ' *' if val_acc > best_val else ''
        print(f'[{phase_label}] Epoch {epoch+1}/{n_epochs} | '
              f'loss: {train_loss:.4f} | '
              f'train: {train_acc:.4f} | '
              f'val: {val_acc:.4f}{star}')

        if val_acc > best_val:
            best_val   = val_acc
            no_improve = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch+1} '
                      f'(best val: {best_val:.4f})')
                break

    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    return history, best_val

In [ ]:
CKPT        = 'best_cloudensenet_14.pt'
all_history = []

# -- Phase 1 -- TopBlock only, LR = 1e-4, 10 epochs --------------------------
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=10, patience=10,
                   checkpoint_path=CKPT, phase_label='Phase 1 (head, 1e-4)')
all_history.append(h)

# -- Phase 2 -- TopBlock only, LR = 1e-5, 10 epochs --------------------------
opt = torch.optim.AdamW(model.classifier.parameters(), lr=1e-5, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=10, patience=10,
                   checkpoint_path=CKPT, phase_label='Phase 2 (head, 1e-5)')
all_history.append(h)

# -- Phase 3 -- unfreeze DenseBlock3 onwards, backbone LR = 5e-5, 20 epochs --
# 'Fine-tune from the 3rd block' (paper §4) = denseblock3, transition3,
# denseblock4, and norm5 in PyTorch's DenseNet121 naming.
unfreeze_from = ['features.denseblock3', 'features.transition3',
                 'features.denseblock4', 'features.norm5']
for name, param in model.named_parameters():
    if any(name.startswith(p) for p in unfreeze_from):
        param.requires_grad = True

backbone_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and not n.startswith('classifier')]
head_params     = list(model.classifier.parameters())

opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 1e-5},
    {'params': backbone_params, 'lr': 5e-5},
], weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 3 (denseblock3+, 5e-5)')
all_history.append(h)

# -- Phase 4 -- same unfrozen scope, LR = 1e-5, 20 epochs --------------------
opt = torch.optim.AdamW([
    {'params': head_params,     'lr': 1e-5},
    {'params': backbone_params, 'lr': 1e-5},
], weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 4 (denseblock3+, 1e-5)')
all_history.append(h)

# -- Phase 5 -- all layers, LR = 1e-6, 20 epochs -----------------------------
for param in model.parameters():
    param.requires_grad = True

opt = torch.optim.AdamW(model.parameters(), lr=1e-6, weight_decay=1e-4)
h, _ = train_phase(model, opt, focal_loss, accuracy,
                   train_loader, valid_loader,
                   n_epochs=20, patience=10,
                   checkpoint_path=CKPT,
                   phase_label='Phase 5 (all, 1e-6)')
all_history.append(h)

In [ ]:
test_acc = evaluate_tm(model, test_loader, accuracy)
print(f'Final test accuracy: {test_acc:.4f}')

In [ ]:
import shutil

drive_save_path = '/content/drive/MyDrive/Colab Notebooks/best_cloudensenet_14.pt'
shutil.copy(CKPT, drive_save_path)
print(f'Checkpoint saved to Drive: {drive_save_path}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']
phase_labels = [
    'Phase 1 (head, 1e-4)',
    'Phase 2 (head, 1e-5)',
    'Phase 3 (block3+, 5e-5)',
    'Phase 4 (block3+, 1e-5)',
    'Phase 5 (all, 1e-6)',
]

offset = 0
for h, label, color in zip(all_history, phase_labels, colors):
    xs = list(range(offset, offset + len(h['train_losses'])))
    axes[0].plot(xs, h['train_losses'], color=color, label=label)
    axes[1].plot(xs, h['train_metrics'], color=color, linestyle='--', alpha=0.5)
    axes[1].plot(xs, h['valid_metrics'],  color=color, label=label)
    for ax in axes:
        ax.axvline(x=offset, color='grey', linewidth=0.5, linestyle=':')
    offset += len(h['train_losses'])

axes[0].set_title('Focal Loss — all 5 phases')
axes[0].set_xlabel('Epoch (cumulative)')
axes[0].set_ylabel('Focal Loss')
axes[0].legend(fontsize=8)

axes[1].set_title('Accuracy — dashed=train, solid=val')
axes[1].set_xlabel('Epoch (cumulative)')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()